In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../data/processed/merged_day1.csv')
df.shape

(36457, 19)

### Convert DAYS_BIRTH and DAYS_EMPLOYED into readable year format

In [4]:
# Convert age to a readable positive number
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)

# Flag the placeholder (365243 = "not currently employed", e.g. pensioners)
df['IS_EMPLOYED'] = (df['DAYS_EMPLOYED'] != 365243).astype(int)

# For employed people, convert to years employed; for unemployed, set to 0
df['YEARS_EMPLOYED'] = np.where(
    df['DAYS_EMPLOYED'] == 365243,
    0,
    (-df['DAYS_EMPLOYED'] / 365).round(1)
)

df[['AGE_YEARS', 'IS_EMPLOYED', 'YEARS_EMPLOYED']].describe()

,AGE_YEARS,IS_EMPLOYED,YEARS_EMPLOYED
count,36457.000000,36457.00000,36457.000000
mean,43.767916,0.83172,6.027973
std,11.508464,0.37412,6.484278
min,20.500000,0.00000,0.000000
25%,34.100000,1.00000,1.100000
50%,42.600000,1.00000,4.300000
75%,53.300000,1.00000,8.600000
max,68.900000,1.00000,43.000000


### Handle Missing Values

In [6]:
print(df.isnull().sum()[df.isnull().sum() > 0])

OCCUPATION_TYPE    11323
dtype: int64


In [7]:
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

### Engineer New Features

In [9]:
# Income per family member — richer signal than raw income alone
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Age buckets — sometimes non-linear age effects show up better as bins
df['AGE_GROUP'] = pd.cut(df['AGE_YEARS'], bins=[0, 25, 35, 45, 55, 100],
                          labels=['<25', '25-35', '35-45', '45-55', '55+'])

# Employment stability ratio — years employed relative to age
df['EMPLOYMENT_RATIO'] = df['YEARS_EMPLOYED'] / df['AGE_YEARS']

# Has children flag — simpler signal than raw count for some models
df['HAS_CHILDREN'] = (df['CNT_CHILDREN'] > 0).astype(int)

### Encode Categorical Variables

In [11]:
# Binary categoricals (Y/N style) — simple mapping
binary_cols = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY']  # adjust to your actual column names
for col in binary_cols:
    df[col] = df[col].map({'Y': 1, 'N': 0})

# Nominal categoricals with few categories — one-hot encoding
low_card_cols = ['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE']
df = pd.get_dummies(df, columns=low_card_cols, drop_first=True)

# High-cardinality categoricals — leave OCCUPATION_TYPE for now, or use
# frequency encoding to avoid too many one-hot columns
df['OCCUPATION_FREQ'] = df['OCCUPATION_TYPE'].map(df['OCCUPATION_TYPE'].value_counts())

### Drop Unnecessary Columns

In [13]:
cols_to_drop = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'OCCUPATION_TYPE', 'AGE_GROUP']

df_model = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [14]:
df_model = pd.get_dummies(df_model, columns=['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE'], drop_first=True)

### Sanity Check

In [16]:
print(df_model.isnull().sum().sum(), "total missing values remaining")
print(df_model.dtypes.value_counts())
df_model.shape

0 total missing values remaining
bool       18
int64      12
float64     6
Name: count, dtype: int64


(36457, 36)

In [17]:
print(df_model.select_dtypes(include=['object']).columns.tolist())  # should be empty list now
print(df_model.dtypes.value_counts())

[]
bool       18
int64      12
float64     6
Name: count, dtype: int64


### Save dataset

In [34]:
df_model.to_csv('../data/processed/model_ready.csv', index=False)